In [1]:
import os
import sys

project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))

if project_root not in sys.path:
    sys.path.append(project_root)

In [15]:
import pipeline.src.python.config as cfg
import pandas as pd
import numpy as np
pd.set_option('display.max_colwidth', None)
import networkx as nx
import leidenalg
import igraph as ig
from bertopic import BERTopic

In [3]:
cluster_df = pd.read_parquet('community_detection_results.parquet')

In [124]:
model_list = ['scopus','science_news','the_guardian']

In [84]:
timeseries_df = pd.DataFrame()

In [85]:
for model in model_list:
    
    cfg_dict = cfg.MAGAZINE_CONFIG[model]
    bertopic_model = BERTopic.load(cfg_dict['REFERENCE_MODEL'])

    model_data = np.load(cfg_dict['OUTPUT_PATH'],allow_pickle=True)

    dataset = pd.read_parquet(cfg_dict['DATASET_PATH'])
    
    tmp_df = bertopic_model.get_document_info(model_data['text'])['CustomName'].reset_index()
    tmp_df = tmp_df.drop(columns='index')

    tmp_df['id'] =  list(model_data['id'])
    tmp_df = tmp_df.rename(columns={'CustomName':'Topic Label'})
    
    tmp_df = tmp_df.merge(cluster_df,on='Topic Label')

    columns_to_mantain = ['id','publicationDate']
    columns_to_drop = [column for column in dataset.columns if column not in columns_to_mantain ]

    tmp_df = tmp_df.merge(dataset.drop(columns=columns_to_drop),on='id')

    timeseries_df = pd.concat([timeseries_df,tmp_df])


2026-02-13 18:54:40,105 - BERTopic - WARNING: You are loading a BERTopic model without explicitly defining an embedding model. If you want to also load in an embedding model, make sure to use `BERTopic.load(my_model, embedding_model=my_embedding_model)`.
2026-02-13 18:55:26,606 - BERTopic - WARNING: You are loading a BERTopic model without explicitly defining an embedding model. If you want to also load in an embedding model, make sure to use `BERTopic.load(my_model, embedding_model=my_embedding_model)`.
2026-02-13 18:55:35,695 - BERTopic - WARNING: You are loading a BERTopic model without explicitly defining an embedding model. If you want to also load in an embedding model, make sure to use `BERTopic.load(my_model, embedding_model=my_embedding_model)`.


In [87]:
timeseries_df['Model'].value_counts()

Model
scopus          62512
the_guardian    14692
science_news      883
Name: count, dtype: int64

In [233]:
timeseries_df[ timeseries_df['Cluster'] == 21 ]

,Topic Label,id,Cluster,Model,publicationDate
581,COVID-19 Mortality and Severe Disease Outcomes,https://api.elsevier.com/content/abstract/scopus_id/85095742265,21,scopus,2020-01-01 00:00:00
599,COVID-19 Mortality and Severe Disease Outcomes,https://api.elsevier.com/content/abstract/scopus_id/85095788917,21,scopus,2020-10-01 00:00:00
605,COVID-19 Mortality and Severe Disease Outcomes,https://api.elsevier.com/content/abstract/scopus_id/85095801041,21,scopus,2020-11-06 00:00:00
718,COVID-19 Exposure and Infection in Healthcare Workers,https://api.elsevier.com/content/abstract/scopus_id/85139208558,21,scopus,2022-09-23 00:00:00
738,Coronavirus Disease Outbreak,https://api.elsevier.com/content/abstract/scopus_id/85139259672,21,scopus,2022-01-01 00:00:00
...,...,...,...,...,...
627,Coronavirus Outbreak Spread,https://www.sciencenews.org/article/morbid-mystery-tour-epidemic-china-encircling-globe,21,science_news,2003-03-26 15:30:34
756,Coronavirus Outbreak Spread,https://www.sciencenews.org/article/new-coronavirus-china-infections-very-few-infants-getting-sick,21,science_news,2020-02-14 16:14:41
757,Coronavirus Outbreak Spread,https://www.sciencenews.org/article/coronavirus-disease-outbreak-severity-symptoms,21,science_news,2020-02-07 17:26:34
877,Coronavirus Outbreak Spread,https://www.sciencenews.org/article/coronavirus-questions-covid19-symptoms-deaths-spread,21,science_news,2020-03-04 09:40:14


In [235]:
#cluster_list = [10,19,21,22,25,38]
cluster_list = [21]

In [236]:
timeseries_df_analysis = timeseries_df[ timeseries_df['Cluster'].isin(cluster_list) ]

In [237]:
if len(cluster_list) > 1:
    # Settare un identificativo per Cluster
    timeseries_df_analysis['Graph_label'] = timeseries_df_analysis['Cluster']
else:
    # Settare un identificativo per Topic Label
    timeseries_df_analysis['Graph_label'] = timeseries_df_analysis['Topic Label']


In [238]:
import polars as pl

timeseries_df_analysis_polars = pl.from_pandas(timeseries_df_analysis)

In [239]:
timeseries_df_analysis_polars = timeseries_df_analysis_polars.with_columns(
    pl.col('publicationDate').dt.truncate('1mo').alias('timestamp'),
)

In [240]:
partial_timeseries_df_analysis_polars = timeseries_df_analysis_polars.group_by(['Graph_label','timestamp']).agg(
    pl.len().alias('occurrence')
).sort(by=['timestamp','Graph_label'],descending=False)

In [241]:
dates = pl.date_range(
    start= partial_timeseries_df_analysis_polars.select(pl.col("timestamp").min()).item(),
    end=  partial_timeseries_df_analysis_polars.select(pl.col("timestamp").max()).item(),
    interval= '1mo',
    eager = True
)

In [242]:
topics = ( 
    partial_timeseries_df_analysis_polars.select(
        pl.col('Graph_label').unique().sort()).to_series()
)

In [243]:
grid = (
    pl.DataFrame({'Graph_label':topics}).join(pl.DataFrame({'timestamp':dates}),how='cross')
)

In [244]:
partial_timeseries_df_analysis_polars = partial_timeseries_df_analysis_polars.with_columns(
        pl.col("timestamp").dt.date().alias("timestamp")
)

In [245]:
complete_timeseries_df_analysis_polars  = (
    grid.join(partial_timeseries_df_analysis_polars,on=['Graph_label','timestamp'],how='left')
    .with_columns(pl.col('occurrence').fill_null(0))
    .sort(by=['timestamp','Graph_label'],descending=False)
)

In [246]:
complete_timeseries_df_analysis = complete_timeseries_df_analysis_polars.to_pandas()

In [247]:
import plotly.express as px

fig = px.line(
    complete_timeseries_df_analysis,
    x="timestamp",
    y="occurrence",
    color="Graph_label",
    markers=True,
    title="Topics time series"
)

fig.update_layout(
    # Titolo
    title={
        'text': "Topics time series",
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 20, 'family': 'Arial, sans-serif'}
    },
    
    # Assi
    xaxis_title="Time",
    yaxis_title="# Articles",
    xaxis=dict(
        showgrid=True,
        gridwidth=1,
        gridcolor='lightgray',
        showline=True,
        linewidth=2,
        linecolor='black'
    ),
    yaxis=dict(
        showgrid=True,
        gridwidth=1,
        gridcolor='lightgray',
        showline=True,
        linewidth=2,
        linecolor='black'
    ),
    
    
    legend=dict(
        title="Topics",
        orientation="h",  
        yanchor="top",
        y=-0.15, 
        xanchor="center",
        x=0.5,  
        bgcolor="rgba(255, 255, 255, 0.8)",
        bordercolor="Black",
        borderwidth=1
    ),
    
    
    plot_bgcolor='white',
    paper_bgcolor='white',
    width=1000,
    height=600,
    
    
    margin=dict(l=80, r=80, t=80, b=120),
    
    
    hovermode='x unified'
)


fig.update_traces(
    line=dict(width=2.5),
    marker=dict(size=6)
)

fig.show()